In [ ]:
pip install transformers torch datasets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 547.8/547.8 kB 9.2 MB/s eta 0:00:00
  Using cached nvidia_cuda_nvrtc_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (23.7 MB)
  Using cached nvidia_cuda_runtime_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (823 kB)
  Using cached nvidia_cuda_cupti_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (14.1 MB)
  Using cached nvidia_cudnn_cu12-8.9.2.26-py3-none-manylinux1_x86_64.whl (731.7 MB)
  Using cached nvidia_cublas_cu12-12.1.3.1-py3-none-manylinux1_x86_64.whl (410.6 MB)
  Using cached nvidia_cufft_cu12-11.0.2.54-py3-none-manylinux1_x86_64.whl (121.6 MB)
  Using cached nvidia_curand_cu12-10.3.2.106-py3-none-manylinux1_x86_64.whl (56.5 MB)
  Using cached nvidia_cusolver_cu12-11.4.5.107-py3-none-manylinux1_x86_64.whl (124.2 MB)
  Using cached nvidia_cusparse_cu12-12.1.0.106-py3-none-manylinux1_x86_64.whl (196.0 MB)
  Using cached nvidia_nccl_cu12-2.20.5-py3-none-manylinux2014_x86_64.whl (176.2 MB)
  Using cached nvidia_nvtx_cu12-12.1.105-py3-none-m

In [ ]:
from datasets import load_dataset

# Load the SQuAD dataset
squad_dataset = load_dataset("squad")

# Display the first few entries
print(squad_dataset)

Generating train split:   0%|          | 0/87599 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10570 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 87599
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 10570
    })
})


In [ ]:
from datasets import load_dataset
import random

def create_structured_prompt(dataset, test_example, num_demonstrations=3):
    # Filter dataset to find samples with the same context
    same_context_samples = [sample for sample in dataset['validation'] if sample['context'] == test_example['context'] and sample['id'] != test_example['id']]

    if len(same_context_samples) <= num_demonstrations:
        raise ValueError("Not enough unique samples with the same context to select the required number of demonstrations.")

    demonstrations = random.sample(same_context_samples, num_demonstrations)

    f = "Give me an answer for the following question by selecting one of the 3 possible answers 1, 2, 3 based on the provided context.\n"

    prompt = ''
    for demo in demonstrations:
        prompt += f
        prompt += f"Context: \"{demo['context']}\"\n"
        prompt += f"Question: \"{demo['question']}\"\n"

        remaining_samples = [s for s in same_context_samples if s != demo]
        possible_answers = random.sample(remaining_samples, 2)
        possible_answers.append(demo)

        random.shuffle(possible_answers)

        options = ['1', '2', '3']
        correct_option = None
        prompt += "Possible Answers:\n"
        for idx, sample in enumerate(possible_answers):
            answer_text = sample['answers']['text'][0]
            option = options[idx]
            prompt += f"{option}: {answer_text}\n"
            if sample == demo:
                correct_option = option

        correct_option = random.choice(options)
        prompt += f"Correct answer: {correct_option}\n"

    prompt += f
    prompt += f"Context: \"{test_example['context']}\"\n"
    prompt += f"Question: \"{test_example['question']}\"\n"

    remaining_samples = [s for s in same_context_samples if s != test_example]
    if len(remaining_samples) >= 2:
        possible_answers = random.sample(remaining_samples, 2)
    else:
        possible_answers = remaining_samples[:2]  # Handle case where not enough samples are available
    possible_answers.append(test_example)

    random.shuffle(possible_answers)

    options = ['1', '2', '3']
    correct_option = None
    prompt += "Possible Answers:\n"
    for idx, sample in enumerate(possible_answers):
        answer_text = sample['answers']['text'][0]
        option = options[idx]
        prompt += f"{option}: {answer_text}\n"
        if sample == test_example:
                correct_option = option

    prompt += "Correct answer: "

    return prompt, correct_option


In [ ]:
from datasets import load_dataset
import random

def create_structured_prompt_random(dataset, test_example, num_demonstrations=3):
    # Filter dataset to find samples with the no same context
    no_same_context_samples = [sample for sample in dataset['validation'] if sample['id'] != test_example['id']]

    if len(no_same_context_samples) <= num_demonstrations:
        raise ValueError("Not enough unique samples with the no same context to select the required number of demonstrations.")

    demonstrations = random.sample(no_same_context_samples, num_demonstrations)

    f = "Give me an answer for the following question by selecting one of the 3 possible answers 1, 2, 3 based on the provided context.\n"

    prompt = ''
    for demo in demonstrations:
        prompt += f
        #prompt += f"Context: \"{demo['context']}\"\n"
        prompt += f"Question: \"{demo['question']}\"\n"

        remaining_samples = [s for s in no_same_context_samples if s != demo]
        possible_answers = random.sample(remaining_samples, 2)
        possible_answers.append(demo)

        random.shuffle(possible_answers)

        options = ['1', '2', '3']
        correct_option = None
        prompt += "Possible Answers:\n"
        for idx, sample in enumerate(possible_answers):
            answer_text = sample['answers']['text'][0]
            option = options[idx]
            prompt += f"{option}: {answer_text}\n"
            if sample == demo:
                correct_option = option
        correct_option = random.choice(options)
        prompt += f"Correct answer: {correct_option}\n"

    prompt += f
    #prompt += f"Context: \"{test_example['context']}\"\n"
    prompt += f"Question: \"{test_example['question']}\"\n"

    remaining_samples = [s for s in no_same_context_samples if s != test_example]
    if len(remaining_samples) >= 2:
        possible_answers = random.sample(remaining_samples, 2)
    else:
        possible_answers = remaining_samples[:2]  # Handle case where not enough samples are available
    possible_answers.append(test_example)

    random.shuffle(possible_answers)

    options = ['1', '2', '3']
    correct_option = None
    prompt += "Possible Answers:\n"
    for idx, sample in enumerate(possible_answers):
        answer_text = sample['answers']['text'][0]
        option = options[idx]
        prompt += f"{option}: {answer_text}\n"
        if sample == test_example:
                correct_option = option

    prompt += "Correct answer: "

    return prompt, correct_option


In [ ]:
# Example usage:
# Load the SQuAD dataset
dataset = load_dataset("squad")

# Assume test_example is given (for illustration purposes, we take the first example from the validation set)
test_example = dataset['validation'][0]

# Create a structured prompt
try:
    prompt = create_structured_prompt_random(dataset, test_example, num_demonstrations=3)
    # Display the prompt
    print(prompt)
except ValueError as e:
    print(e)

('Give me an answer for the following question by selecting one of the 3 possible answers 1, 2, 3 based on the provided context.\nQuestion: "Which NASA orbiter photographed evidence of each site on the moon that a manned Apollo mission landing occurred?"\nPossible Answers:\n1: $5 million\n2: Lunar Reconnaissance Orbiter\n3: a pointless pursuit\nCorrect answer: 1\nGive me an answer for the following question by selecting one of the 3 possible answers 1, 2, 3 based on the provided context.\nQuestion: "From what pad was Apollo 5 launched from?"\nPossible Answers:\n1: 1775\n2: pad 37\n3: a mouth that can usually be closed by muscles; a pharynx ("throat"); a wider area in the center that acts as a stomach; and a system of internal canals.\nCorrect answer: 3\nGive me an answer for the following question by selecting one of the 3 possible answers 1, 2, 3 based on the provided context.\nQuestion: "ABC created what company as a syndication distributor in response to the FCC\'s fin-syn rules?"\n

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import login

import os

# Add your Hugging Face token here
# os.environ["HF_TOKEN"] = ""
login(token=os.environ["HF_TOKEN"])

# Load the tokenizer and model
tokenizer = AutoTokenizer.from_pretrained("google/gemma-2b", use_auth_token=os.environ["HF_TOKEN"])
model = AutoModelForCausalLM.from_pretrained("google/gemma-2b", use_auth_token=os.environ["HF_TOKEN"])

model.to('cuda')

The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `huggingface-cli` if you want to set the git credential as well.
Token is valid (permission: read).
Your token has been saved to /root/.cache/huggingface/token
Login successful


/usr/local/lib/python3.10/dist-packages/transformers/models/auto/tokenization_auto.py:769: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/33.6k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/67.1M [00:00<?, ?B/s]

`config.hidden_act` is ignored, you should use `config.hidden_activation` instead.
Gemma's activation function will be set to `gelu_pytorch_tanh`. Please, use
`config.hidden_activation` if you want to override this behaviour.
See https://github.com/huggingface/transformers/pull/29402 for more details.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

GemmaForCausalLM(
  (model): GemmaModel(
    (embed_tokens): Embedding(256000, 2048, padding_idx=0)
    (layers): ModuleList(
      (0-17): 18 x GemmaDecoderLayer(
        (self_attn): GemmaSdpaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=256, bias=False)
          (v_proj): Linear(in_features=2048, out_features=256, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (rotary_emb): GemmaRotaryEmbedding()
        )
        (mlp): GemmaMLP(
          (gate_proj): Linear(in_features=2048, out_features=16384, bias=False)
          (up_proj): Linear(in_features=2048, out_features=16384, bias=False)
          (down_proj): Linear(in_features=16384, out_features=2048, bias=False)
          (act_fn): PytorchGELUTanh()
        )
        (input_layernorm): GemmaRMSNorm()
        (post_attention_layernorm): GemmaRMSNorm()
      )
    )
    (norm): GemmaR

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
import torch
from collections import defaultdict

def evaluate_model(dataset, num_examples=100,num_dem = 1):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)

    # We'll assume dataset is structured with 'validation' split containing the required fields
    #validation_set = dataset['validation']

    # Randomly select a subset of the validation dataset for evaluation
    #test_examples = random.sample(dataset['validation'], 100)
    random.seed(42)
    test_examples = random.sample(list(dataset['validation']), num_examples)

    correct_answers = 0
    total = 0
    invalid_answers = 0

    for test_example in test_examples:
        try:
            # Generate the structured prompt for the model
            prompt, correct_option = create_structured_prompt(dataset, test_example,num_dem)

            # Encode the prompt to input_ids that can be used by the model
            inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=2048).to(device)

            outputs = model.generate(
            inputs.input_ids,
            max_new_tokens=1,
            temperature=1,
            num_return_sequences=1,
            do_sample=True
            )

            generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

            lines = generated_text.splitlines()[-3:]

            last_line = ""
            for line_index, line in enumerate(lines):
              if line.startswith("Correct answer:"):
                last_line = line[15:].strip()
                if not last_line and line_index + 1 < len(lines):
                  next_line = lines[line_index + 1].strip()
                  if next_line in ['1','2','3']:
                      last_line = next_line
                break

            # Check if predicted token is a valid option (a, b, or c)
            if last_line in ['1', '2', '3']:
                #correct_option = prompt.split("Which is the correct answer?")[-1].strip().split('\n')[0]
                if last_line == correct_option:
                    correct_answers += 1
            else:
              invalid_answers +=1

            total += 1

        except Exception as e:
          continue


    accuracy = correct_answers / total if total > 0 else 0
    invalid_per = (invalid_answers/total)*100 if total > 0 else 0
    print(f"Accuracy: {accuracy:.2f}")
    print(f"Invalid answers percentage: {invalid_per:.2f}%")

    return accuracy,invalid_per

# To use the evaluate_model function, ensure you have the appropriate dataset loaded with the required fields
# dataset = load_dataset('some_dataset_name_here')
# evaluate_model(dataset)









In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
import torch
from collections import defaultdict

def evaluate_model_random(dataset, num_examples=100,num_dem = 1):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)

    # We'll assume dataset is structured with 'validation' split containing the required fields
    #validation_set = dataset['validation']

    # Randomly select a subset of the validation dataset for evaluation
    #test_examples = random.sample(dataset['validation'], 100)
    test_examples = random.sample(list(dataset['validation']), num_examples)

    correct_answers = 0
    total = 0
    invalid_answers = 0

    for test_example in test_examples:
        try:
            # Generate the structured prompt for the model
            prompt, correct_option = create_structured_prompt_random(dataset, test_example,num_dem)

            # Encode the prompt to input_ids that can be used by the model
            inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=2048).to(device)

            outputs = model.generate(
            inputs.input_ids,
            max_new_tokens=1,
            temperature=1,
            num_return_sequences=1,
            do_sample=True
            )

            generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

            lines = generated_text.splitlines()[-3:]

            last_line = ""
            for line_index, line in enumerate(lines):
              if line.startswith("Correct answer:"):
                last_line = line[15:].strip()
                if not last_line and line_index + 1 < len(lines):
                  next_line = lines[line_index + 1].strip()
                  if next_line in ['1','2','3']:
                      last_line = next_line
                break

            # Check if predicted token is a valid option (a, b, or c)
            if last_line in ['1', '2', '3']:
                #correct_option = prompt.split("Which is the correct answer?")[-1].strip().split('\n')[0]
                if last_line == correct_option:
                    correct_answers += 1
            else:
              invalid_answers +=1

            total += 1

        except Exception as e:
          continue


    accuracy = correct_answers / total if total > 0 else 0
    invalid_per = (invalid_answers/total)*100 if total > 0 else 0
    print(f"Accuracy: {accuracy:.2f}")
    print(f"Invalid answers percentage: {invalid_per:.2f}%")

    return accuracy,invalid_per

# To use the evaluate_model function, ensure you have the appropriate dataset loaded with the required fields
# dataset = load_dataset('some_dataset_name_here')
# evaluate_model(dataset)









In [ ]:
evaluate_model(dataset,1000,1)

Accuracy: 0.39
Invalid answers percentage: 0.11%


(0.39384288747346075, 0.10615711252653928)

In [ ]:
evaluate_model(dataset,1000,0)

Accuracy: 0.34
Invalid answers percentage: 1.50%


(0.34434434434434436, 1.5015015015015014)

In [ ]:
evaluate_model(dataset,1000,3)

Accuracy: 0.47
Invalid answers percentage: 0.51%


(0.46683673469387754, 0.5102040816326531)

Random answers in demonstrations:


In [ ]:
evaluate_model(dataset,1000,1)

Accuracy: 0.43
Invalid answers percentage: 0.32%


(0.4263157894736842, 0.3157894736842105)

In [ ]:
evaluate_model(dataset,1000,3)

Accuracy: 0.46
Invalid answers percentage: 0.51%


(0.45897435897435895, 0.5128205128205128)

#random context

In [ ]:
evaluate_model_random(dataset,1000,1)

Accuracy: 0.52
Invalid answers percentage: 0.50%


(0.522, 0.5)

In [ ]:
evaluate_model_random(dataset,1000,3)

Accuracy: 0.58
Invalid answers percentage: 0.30%


(0.578, 0.3)

#random context random answers

In [ ]:
evaluate_model_random(dataset,1000,1)

Accuracy: 0.42
Invalid answers percentage: 0.20%


(0.425, 0.2)

In [ ]:
evaluate_model_random(dataset,1000,3)

Accuracy: 0.47
Invalid answers percentage: 0.10%


(0.471, 0.1)

Without context:

In [ ]:
evaluate_model_random(dataset,1000,0)

Accuracy: 0.40
Invalid answers percentage: 2.30%


(0.398, 2.3)

In [ ]:
evaluate_model_random(dataset,1000,1)

Accuracy: 0.46
Invalid answers percentage: 0.00%


(0.456, 0.0)

In [ ]:
evaluate_model_random(dataset,1000,3)

Accuracy: 0.49
Invalid answers percentage: 0.10%


(0.495, 0.1)